# Stable Diffusion Web UI on Google Colab

このノートブックは、Google Colab上でStable Diffusion v1.5を使用した画像生成環境を提供します。

**重要: GPU設定が必須です**
1. メニューから「ランタイム」→「ランタイムのタイプを変更」を選択
2. 「ハードウェア アクセラレータ」を **GPU** に変更
3. 「保存」をクリック

**使用モデル:** runwayml/stable-diffusion-v1-5 (fp16)

**スケジューラ:** EulerDiscreteScheduler

In [ ]:
# Cell 1: Environment Setup & GPU Verification
# GPU確認と必要なライブラリのインストール

import subprocess
import sys

# GPU確認
print("=" * 50)
print("GPU確認")
print("=" * 50)
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GPU検出成功!")
    print(result.stdout)
else:
    print("\n⚠️ 警告: GPUが検出されませんでした")
    print("\n解決方法:")
    print("1. メニューから [ランタイム] → [ランタイムのタイプを変更] を選択")
    print("2. [ハードウェア アクセラレータ] を [GPU] に変更")
    print("3. [保存] をクリックしてランタイムを再起動")
    print("4. このセルを再実行してください\n")
    raise RuntimeError("GPU not available. Please enable GPU runtime.")

print("\n" + "=" * 50)
print("PyTorch CUDA対応版のインストール")
print("=" * 50)

# CUDA対応のPyTorchを明示的にインストール
print("Installing PyTorch with CUDA support...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu118"
])

# その他の必要なライブラリをインストール
print("\nInstalling other required packages...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers", "transformers", "accelerate", 
    "scipy", "safetensors", "gradio"
])

# PyTorchのCUDA確認
import torch
print("\n" + "=" * 50)
print("PyTorch CUDA確認")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print("\n✓ 環境セットアップ完了! 次のセルに進んでください。")
else:
    print("\n❌ エラー: PyTorchでCUDAが有効になっていません")
    print("\nトラブルシューティング:")
    print("1. ランタイムを再起動: [ランタイム] → [ランタイムを再起動]")
    print("2. GPU設定を確認: [ランタイム] → [ランタイムのタイプを変更] → [GPU]")
    print("3. このセルを再実行")
    raise RuntimeError("Torch not compiled with CUDA enabled")

In [ ]:
# Cell 2: Import & Model Loading
# ライブラリのインポートとモデルのロード

import torch
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
import gradio as gr
from PIL import Image

print("=" * 50)
print("Stable Diffusion モデルのロード")
print("=" * 50)

# CUDA確認（念のため）
if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ CUDA is not available. Please run Cell 1 again or check GPU settings."
    )

print(f"✓ CUDA available: {torch.cuda.is_available()}")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")
print(f"✓ PyTorch version: {torch.__version__}")

# モデルID
model_id = "runwayml/stable-diffusion-v1-5"
print(f"\nLoading model: {model_id}")
print("(初回実行時は数分かかる場合があります...)\n")

# EulerDiscreteSchedulerのセットアップ
scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")
print("✓ Scheduler loaded")

# StableDiffusionPipelineをfp16でロード
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    scheduler=scheduler,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
print("✓ Pipeline loaded")

# GPUへ転送
pipe = pipe.to("cuda")
print("✓ Model transferred to GPU")

print("\n" + "=" * 50)
print("✓ モデルのロード完了!")
print("=" * 50)
print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print("\n次のセルに進んでください。")

In [ ]:
# Cell 3: Generation Logic
# 画像生成関数の定義

def generate_image(prompt, negative_prompt, steps, guidance):
    """
    Stable Diffusionを使用して画像を生成します。
    
    Args:
        prompt (str): 生成したい画像の説明（必須）
        negative_prompt (str): 避けたい要素の説明（オプション）
        steps (int): 推論ステップ数（1-100）
        guidance (float): ガイダンススケール（1-20）
    
    Returns:
        PIL.Image: 生成された画像
    """
    try:
        print(f"\n{'='*50}")
        print(f"Generating image...")
        print(f"{'='*50}")
        print(f"Prompt: '{prompt}'")
        if negative_prompt:
            print(f"Negative Prompt: '{negative_prompt}'")
        print(f"Steps: {steps}, Guidance Scale: {guidance}")
        
        # 画像生成
        with torch.no_grad():
            result = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt if negative_prompt else None,
                num_inference_steps=int(steps),
                guidance_scale=guidance,
                height=512,
                width=512
            )
        
        image = result.images[0]
        print(f"\n✓ Image generated successfully!")
        print(f"{'='*50}\n")
        return image
        
    except Exception as e:
        print(f"\n❌ Error during generation: {e}")
        print(f"{'='*50}\n")
        # エラー時は空白画像を返す
        return Image.new('RGB', (512, 512), color='gray')

print("✓ Generation function defined successfully!")
print("\n次のセルでUIを起動します。")

In [ ]:
# Cell 4: UI Launch
# Gradio UIの構築と起動

print("=" * 50)
print("Gradio UIの起動")
print("=" * 50)

# Gradio Interfaceの作成
interface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(
            label="Prompt (プロンプト)",
            placeholder="例: a beautiful landscape with mountains and lake, sunset, highly detailed",
            lines=3
        ),
        gr.Textbox(
            label="Negative Prompt (ネガティブプロンプト)",
            placeholder="例: blurry, bad quality, distorted",
            lines=2,
            value=""
        ),
        gr.Slider(
            minimum=1,
            maximum=100,
            value=50,
            step=1,
            label="Inference Steps (推論ステップ数)"
        ),
        gr.Slider(
            minimum=1,
            maximum=20,
            value=7.5,
            step=0.5,
            label="Guidance Scale (ガイダンススケール)"
        )
    ],
    outputs=gr.Image(label="Generated Image (生成画像)", type="pil"),
    title="🎨 Stable Diffusion Web UI",
    description="Stable Diffusion v1.5を使用した画像生成インターフェース。プロンプトを入力して画像を生成できます。",
    examples=[
        [
            "a photo of an astronaut riding a horse on mars",
            "blurry, bad quality",
            50,
            7.5
        ],
        [
            "a beautiful japanese garden with cherry blossoms, spring, highly detailed",
            "ugly, distorted, low quality",
            50,
            7.5
        ],
        [
            "cyberpunk city at night, neon lights, futuristic, 4k",
            "blurry, bad anatomy",
            50,
            7.5
        ]
    ],
    cache_examples=False
)

# UIを起動（share=Trueでパブリックアクセス可能）
print("\nLaunching Gradio UI...")
print("(起動には数秒かかります...)\n")

interface.launch(share=True, debug=True)

print("\n" + "=" * 50)
print("✓ Web UI起動完了!")
print("=" * 50)
print("\n上に表示されている2つのURLから選んでアクセスしてください:")
print("1. Running on local URL: Colab内でのみアクセス可能")
print("2. Running on public URL: Colab外部からもアクセス可能 (share=True)")
print("\n画像生成を楽しんでください!")